In [ ]:
import re
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder


In [3]:
BASE_DIR = Path.cwd().parent

STAGE1_PATH = (
    BASE_DIR
    / "data"
    / "processed"
    / "stage1"
    / "stage1_model_keys.csv"
)

print("Current directory:", Path.cwd())
print("Stage 1 file:", STAGE1_PATH.resolve())
print("File exists:", STAGE1_PATH.exists())

Current directory: e:\DEPI-Grad\DEPI-GRADUATION-PROJECT\notebooks
Stage 1 file: E:\DEPI-Grad\DEPI-GRADUATION-PROJECT\data\processed\stage1\stage1_model_keys.csv
File exists: True


In [4]:
if not STAGE1_PATH.exists():
    raise FileNotFoundError(
        f"Stage 1 file not found: {STAGE1_PATH.resolve()}"
    )

stage1_df = pd.read_csv(STAGE1_PATH)

print("Shape:", stage1_df.shape)
print("Columns:", stage1_df.columns.tolist())

display(stage1_df.head())

Shape: (5258, 36)
Columns: ['title', 'price', 'location', 'ad_url', 'ad_id', 'run_id', 'brand', 'storage_gb', 'ram_gb', 'storage_src', 'condition', 'condition_src', 'ram_src', 'has_5g', 'has_warranty', 'has_box', 'mentions_exchange', 'is_wanted_ad', 'is_accessory_hint', 'is_tablet_hint', 'is_parts_hint', 'accessory_first', 'bundle_hint', 'is_feature_phone_hint', 'price_placeholder', 'loc_area', 'loc_city', 'price_missing', 'price_suspicious', 'price_very_high', 'title_normalized', 'title_punct_normalized', 'title_group', 'model_key', 'match_rule', 'confidence']


,title,price,location,ad_url,ad_id,run_id,brand,storage_gb,ram_gb,storage_src,...,loc_city,price_missing,price_suspicious,price_very_high,title_normalized,title_punct_normalized,title_group,model_key,match_rule,confidence
0,Honor Magic V5,80000.0,اوبن اير مول، مدينتي,/ad/honor-magic-v5-ID209010621.html,209010621,legacy,honor,NaN,NaN,NaN,...,مدينتي,False,False,False,honor magic v5,honor magic v5,honor magic v5,honor magic v5,honor:rule_04,NaN
1,شاومي 17 الترا xiaomi 17 ultra,70000.0,التجمع الخامس، القاهرة الجديدة,/ad/%D8%B4%D8%A7%D9%88%D9%85%D9%8A-17-%D8%A7%D...,208281312,legacy,xiaomi,NaN,NaN,NaN,...,القاهرة الجديدة,False,False,False,شاومي17الترا xiaomi17ultra,شاومي17الترا xiaomi17ultra,شاومي17الترا xiaomi17ultra,xiaomi 17 ultra,xiaomi:rule_17,NaN
2,Apple - iPhone 17 Pro,75000.0,محور 26 يوليو، 6 اكتوبر,/ad/apple-iphone-17-pro-ID208973056.html,208973056,legacy,apple,NaN,NaN,NaN,...,6 اكتوبر,False,False,False,apple iphone17pro,apple iphone17pro,apple iphone17pro,apple iphone 17 pro,apple:model_pattern_06,1.0
3,Honor 500 pro,35000.0,6 اكتوبر، الجيزة,/ad/honor-500-pro-ID208029599.html,208029599,legacy,honor,NaN,NaN,NaN,...,الجيزة,False,False,False,honor500pro,honor500pro,honor500pro,honor 500 pro,honor:rule_07,NaN
4,آبل - آيفون 15 برو ماكس,58000.0,ستيلا هليوبوليس، مدينة الشروق,/ad/%D8%A2%D8%A8%D9%84-%D8%A2%D9%8A%D9%81%D9%8...,208989761,legacy,apple,NaN,NaN,NaN,...,مدينة الشروق,False,False,False,ابل ايفون15برو ماكس,ابل ايفون15برو ماكس,ابل ايفون15برو ماكس,apple iphone 15 pro max,apple:model_pattern_02,1.0


In [ ]:
display(stage1_df.dtypes.to_frame("dtype"))
display(stage1_df.isna().sum().to_frame("missing_values"))

,dtype
title,object
price,float64
location,object
ad_url,object
ad_id,int64
run_id,object
brand,object
storage_gb,float64
ram_gb,float64
storage_src,object


,missing_values
title,0
price,0
location,4
ad_url,0
ad_id,0
run_id,0
brand,279
storage_gb,4891
ram_gb,5146
storage_src,4891


In [ ]:
known_flags = [
    "storage_known",
    "ram_known",
    "condition_known",
]

for col in known_flags:
    print(f"{col}: {'EXISTS' if col in stage1_df.columns else 'MISSING'}")

storage_known: MISSING
ram_known: MISSING
condition_known: MISSING


In [7]:
spec_cols = ["storage_gb", "ram_gb", "condition"]

display(
    pd.DataFrame({
        "dtype": stage1_df[spec_cols].dtypes,
        "missing_count": stage1_df[spec_cols].isna().sum(),
        "known_count": stage1_df[spec_cols].notna().sum(),
        "known_percentage": (
            stage1_df[spec_cols].notna().mean() * 100
        ).round(2),
    })
)

,dtype,missing_count,known_count,known_percentage
storage_gb,float64,4891,367,6.98
ram_gb,float64,5146,112,2.13
condition,object,4975,283,5.38


creating known flags  (storage,ram,condition)

In [8]:
stage1_df["storage_known"] = stage1_df["storage_gb"].notna()
stage1_df["ram_known"] = stage1_df["ram_gb"].notna()
stage1_df["condition_known"] = stage1_df["condition"].notna()

In [9]:
known_flags = [
    "storage_known",
    "ram_known",
    "condition_known",
]

display(
    stage1_df[known_flags].agg(["sum", "count"]).T
)

display(
    stage1_df[known_flags].mean().mul(100).round(2)
    .to_frame("known_percentage")
)

,sum,count
storage_known,367,5258
ram_known,112,5258
condition_known,283,5258


,known_percentage
storage_known,6.98
ram_known,2.13
condition_known,5.38


In [10]:
print("Shape:", stage1_df.shape)
print("Unique ad IDs:", stage1_df["ad_id"].nunique())
print("Unique title groups:", stage1_df["title_group"].nunique())

Shape: (5258, 39)
Unique ad IDs: 5258
Unique title groups: 2191


In [ ]:
display(stage1_df.dtypes.to_frame("dtype"))

,dtype
title,object
price,float64
location,object
ad_url,object
ad_id,int64
run_id,object
brand,object
storage_gb,float64
ram_gb,float64
storage_src,object


In [12]:
stage1_df['brand'].value_counts()

brand
apple       2416
samsung      984
xiaomi       380
oppo         325
realme       187
honor        181
infinix      144
vivo         131
huawei       114
motorola      24
oneplus       18
zte           17
google        17
tecno         15
sony           8
itel           7
nokia          5
lenovo         4
lg             2
Name: count, dtype: int64

In [13]:
stage1_df['model_key'].value_counts()

model_key
unknown                    562
apple iphone 13            329
apple iphone 13 pro max    175
apple iphone 11 pro max    140
apple iphone 12 pro max    127
                          ... 
zte red magic 7              1
infinix smart zero           1
tecno spark 30c              1
samsung grand                1
honor 30                     1
Name: count, Length: 728, dtype: int64

In [ ]:
# Create an independent copy for Stage 2
stage2_df = stage1_df.copy(deep=True)
print("Stage 1 shape:", stage1_df.shape)
print("Stage 2 shape:", stage2_df.shape)
print("Are they the same object?", stage1_df is stage2_df)

Stage 1 shape: (5258, 39)
Stage 2 shape: (5258, 39)
Are they the same object? False


In [ ]:
categorical_features = ["model_key", "brand"]

known_features = [
    "storage_known",
    "ram_known",
    "condition_known",
]

feature_columns = categorical_features + known_features
train_df = stage2_df.loc[stage2_df["price"].notna()].copy()
X = train_df[feature_columns].copy()
y = train_df["price"].copy()
groups = train_df["title_group"].copy()

In [16]:
print("Training rows:", len(train_df))
print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Missing target values:", y.isna().sum())
print("Missing title groups:", groups.isna().sum())

display(X.head())

Training rows: 5258
Features shape: (5258, 5)
Target shape: (5258,)
Missing target values: 0
Missing title groups: 0


,model_key,brand,storage_known,ram_known,condition_known
0,honor magic v5,honor,False,False,False
1,xiaomi 17 ultra,xiaomi,False,False,False
2,apple iphone 17 pro,apple,False,False,False
3,honor 500 pro,honor,False,False,False
4,apple iphone 15 pro max,apple,False,False,False


In [ ]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, categorical_features),
    ("known_flags", "passthrough", known_features)
])

print("Preprocessor is ready.")
print("Categorical features:", categorical_features)
print("Known flags:", known_features)

Preprocessor created successfully.
Categorical features: ['model_key', 'brand']
Known flags: ['storage_known', 'ram_known', 'condition_known']


In [23]:
# Check required columns
missing_columns = [col for col in feature_columns if col not in stage1_df.columns]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

# Check that X, y, and groups have matching rows
if not (X.index.equals(y.index) and X.index.equals(groups.index)):
    raise ValueError("X, y, and groups have mismatched rows")

# Review the data
print("Features:", X.columns.tolist())
print("\nMissing values:\n", X.isna().sum())
print("\nTarget summary:\n", y.describe())
print("\nUnique title groups:", groups.nunique())

Features: ['model_key', 'brand', 'storage_known', 'ram_known', 'condition_known']

Missing values:
 model_key            0
brand              279
storage_known        0
ram_known            0
condition_known      0
dtype: int64

Target summary:
 count      5258.000000
mean      22858.786421
std       20722.258158
min        1000.000000
25%        7000.000000
50%       16500.000000
75%       32000.000000
max      185500.000000
Name: price, dtype: float64

Unique title groups: 2191


In [20]:
X_transformed = preprocessor.fit_transform(X)

print("Original shape:", X.shape)
print("Encoded shape:", X_transformed.shape)
print("Encoding successful!")

Original shape: (5258, 5)
Encoded shape: (5258, 751)
Encoding successful!
